# 数据处理

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection  import train_test_split
import xgboost as xgb
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
# 读入数据
raw_data= pd.read_csv('保险数据_第一期.csv')
# 缺失值(缺失值占比很少，直接删除)
df= raw_data.dropna().reset_index(drop=True)
# 分割标签
labels= df.pop("resp_flag")
# 数据类型拆分
cat_cols= df.select_dtypes(include= ["object"])   # 分类型
num_cols= df.select_dtypes(exclude= ["object"])   # 数值型
# 数据编码
encoder= OrdinalEncoder()
cat_trans= encoder.fit_transform(cat_cols)
df_cat= pd.DataFrame(cat_trans, columns= cat_cols.columns)
# 数据合并
data= pd.concat([df_cat, num_cols], axis=1)
# 分割测试集
x_train, x_test, y_train, y_test= train_test_split(data, labels, test_size= 0.3, stratify= labels, random_state= 42)

# Tuning the hyper-parameters（调参）

## Grid Search（网格搜索）

In [3]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param = {"n_estimators": np.arange(100,150,20),    # 启，止，步长
         "max_depth": np.arange(3,6,2),
        "subsample": [0.8, 0.9],
        "colsample_bytree": [0.8, 0.9]}

xgb = XGBClassifier()
grid_search_xgb = GridSearchCV(xgb, param, cv=3, n_jobs=-1)  
grid_search_xgb.fit(x_train, y_train)

print("AUC: ", roc_auc_score(y_test, grid_search_xgb.predict_proba(x_test)[:, 1]))
grid_search_xgb.best_params_ #最优参数

GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None,...
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None,
                                     random_state=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 0.9],
                         'max_depth': array([3, 5]),
                         'n_estimators': array([100, 120, 140]),
                         'subsample': [0.8, 0.9]})

AUC:  0.7046278103430083


{'colsample_bytree': 0.8,
 'max_depth': 3,
 'n_estimators': 120,
 'subsample': 0.9}

## Halving Search

In [3]:
# Halving 搜索
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingGridSearchCV

param = {"n_estimators": np.arange(100,150,1),    # 启，止，步长
         "max_depth": np.arange(3,6,1),
        "subsample": np.arange(0.5,1,0.1),
        "colsample_bytree": np.arange(0.5,1,0.1)}

xgb = XGBClassifier()
halving_search_xgb = HalvingGridSearchCV(xgb, param, factor=3, cv=3, verbose=1, n_jobs=-1)  
halving_search_xgb.fit(x_train, y_train)

print("AUC: ", roc_auc_score(y_test, halving_search_xgb.predict_proba(x_test)[:, 1]))
halving_search_xgb.best_params_ #最优参数

n_iterations: 8
n_required_iterations: 8
n_possible_iterations: 8
min_resources_: 12
max_resources_: 27342
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 3750
n_resources: 12
Fitting 3 folds for each of 3750 candidates, totalling 11250 fits
----------
iter: 1
n_candidates: 1250
n_resources: 36
Fitting 3 folds for each of 1250 candidates, totalling 3750 fits
----------
iter: 2
n_candidates: 417
n_resources: 108
Fitting 3 folds for each of 417 candidates, totalling 1251 fits
----------
iter: 3
n_candidates: 139
n_resources: 324
Fitting 3 folds for each of 139 candidates, totalling 417 fits
----------
iter: 4
n_candidates: 47
n_resources: 972
Fitting 3 folds for each of 47 candidates, totalling 141 fits
----------
iter: 5
n_candidates: 16
n_resources: 2916
Fitting 3 folds for each of 16 candidates, totalling 48 fits
----------
iter: 6
n_candidates: 6
n_resources: 8748
Fitting 3 folds for each of 6 candidates, totalling 18 fits
----------
iter: 7
n_candidates: 2
n

HalvingGridSearchCV(cv=3,
                    estimator=XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rat...
                    param_grid={'colsample_bytree': array([0.5, 0.6, 0.7, 0.8, 0.9]),
                                'max_depth': array([3, 4, 5]),
                                'n_estimators': array([100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
       113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
       126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138,
       139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]),
                                'subsample': array([0.5, 0.6, 0.7, 0.8, 0.9])},
                    verbose=1)

AUC:  0.7027465618515003


{'colsample_bytree': 0.7999999999999999,
 'max_depth': 3,
 'n_estimators': 132,
 'subsample': 0.7999999999999999}

## Randomized Search（随机搜索）

In [4]:
# 随机搜索
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

param = dict(n_estimators= np.arange(100,150,1),    # 启，止，步长
         max_depth= np.arange(3,6,1),
         subsample= uniform(loc=0.5, scale=0.5),   # 启，增加量
         colsample_bytree= uniform(loc=0.5, scale=0.5))

xgb = XGBClassifier()
random_search_xgb = RandomizedSearchCV(xgb, param, cv=3, n_iter=50, verbose=1, n_jobs=-1)  
random_search_xgb.fit(x_train, y_train)

print("AUC: ", roc_auc_score(y_test, random_search_xgb.predict_proba(x_test)[:, 1]))
random_search_xgb.best_params_ #最优参数

Fitting 3 folds for each of 50 candidates, totalling 150 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric=None, feature_types=None,
                                           gamma=None, grow_policy=None,
                                           importance_type=None,
                                           interaction_constraints=None,
                                           learning_rate...
                                        'max_depth': array([3, 4, 5]),
                                        'n_estimators': array([100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
       113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
       126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138,
       139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]),
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000002DE2E614E30>},
                   verbose=1)

AUC:  0.7063002805339971


{'colsample_bytree': 0.9682668907336596,
 'max_depth': 3,
 'n_estimators': 107,
 'subsample': 0.9835589970988916}

In [5]:
# 查看最优模型参数
random_search_xgb.best_params_

{'colsample_bytree': 0.9682668907336596,
 'max_depth': 3,
 'n_estimators': 107,
 'subsample': 0.9835589970988916}

## Bayesian Optimization（贝叶斯搜索）

In [6]:
# 贝叶斯搜索
import optuna
from optuna.samplers import TPESampler
import lightgbm as lgb

def objective(trial):  # 定义优化函数
    param = {
        'lambda_l2': trial.suggest_float('lambda_l2', 0.01, 10.0),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        "verbosity" : -1,   # 静默输出，训练过程不输出冗余信息
    }
    d_train = lgb.Dataset(x_train, label=y_train)
    d_valid = lgb.Dataset(x_test, label=y_test)
    
    model = lgb.train(params= param, train_set=d_train, valid_sets=d_valid) # 建模
    AUC= roc_auc_score(y_test, model.predict(x_test))   # metric
    return AUC  # AUC作为目标值
sampler = TPESampler()  # 使用基于树结构的贝叶斯优化算法-TPE
lgb_opt = optuna.create_study(sampler=sampler, direction='maximize')  # 最大化目标值
lgb_opt.optimize(
    func=objective,   # 要优化的函数
    n_trials=100,     # 100次迭代
    show_progress_bar=True,   # 显示进度条
    n_jobs=-1         # 使用所有CPU核心并行化
)  

# 打印结果
print("Best_params: ", lgb_opt.best_params)
print("Best score: ", lgb_opt.best_value)

[I 2025-05-04 18:04:09,878] A new study created in memory with name: no-name-d392f760-1853-49ad-9c75-141b336fd0dd


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-05-04 18:04:11,607] Trial 7 finished with value: 0.7089170514457871 and parameters: {'lambda_l2': 0.4851894066721498, 'num_leaves': 43, 'feature_fraction': 0.6387561337828198, 'bagging_fraction': 0.7779389871155624}. Best is trial 7 with value: 0.7089170514457871.
[I 2025-05-04 18:04:12,085] Trial 2 finished with value: 0.7100012856292124 and parameters: {'lambda_l2': 7.598199721728203, 'num_leaves': 49, 'feature_fraction': 0.6416713940029101, 'bagging_fraction': 0.9986894799731324}. Best is trial 2 with value: 0.7100012856292124.
[I 2025-05-04 18:04:12,719] Trial 6 finished with value: 0.7050375136901318 and parameters: {'lambda_l2': 1.0910014798246075, 'num_leaves': 90, 'feature_fraction': 0.9212601074447956, 'bagging_fraction': 0.40928136909599355}. Best is trial 2 with value: 0.7100012856292124.
[I 2025-05-04 18:04:14,060] Trial 8 finished with value: 0.7068756299279926 and parameters: {'lambda_l2': 9.971685358591744, 'num_leaves': 93, 'feature_fraction': 0.9427174047285386